> ##### 해당 문서에서 다른 환경에서 최초 1회 실행.
> 1. JSON 읽기
> 2. "질병명, 증상, 피부 부위"를 메타데이터로 구성
> 3. "질병 정보"만 쪼개기
>     - "질병 정보"는 문자열이 평균 1만자임.
>     - 토큰수 초과로 답변을 생성하지 못할 수 있고
>     - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림.
> 4. 쪼갠 문서를 임베딩해서 -> 벡터 데이터베이스에 저장

> ##### FastAPI에서 반복 실행
> 4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
> 5. 유사도 검색으로 가져온 문서를 llm에 질문과 함께 전달 

### 1. json 읽기

In [1]:
%pip install --upgrade --quiet docx2txt langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [16]:
# 1. JSON 파일 로드
with open("./all_diseases_information.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [17]:
for doc in data[:1]:
    print(doc)

{'disease': "Athlete's foot", 'symptoms': ['itchiness', 'peeling skin', 'red rash', 'cracked skin', 'bleeding'], 'skin_site': ['between toes', 'feet'], 'disease_information': "Athlete's foot is a type of fungal infection. If your baby has it, you may notice peeling skin between their toes. In more severe cases, the skin on their feet may crack and bleed. The skin can look red on white skin, but this may be less noticeable on brown or black skin. It can also be itchy for your little one, so you may notice them scratching their feet more than usual.\nIf you think your baby has athlete's foot, take them to your local pharmacist. They'll be able to prescribe a cream to clear up the infection.\nAthlete's foot is a fungal infection that causes a red, itchy, moist rash, usually between the toes. It's rare in toddlers, but it may be more likely to happen if you take your child swimming a lot. This is because the fungus thrives in warm, damp areas like showers and changing rooms.\nYou can help 

### 2. "질병명, 증상, 피부 부위"를 메타데이터로 구성

In [18]:
# 2. disease_information만 임베딩하면서 metadata는 따로 구성
documents = []
for entry in data:
    disease_info = entry["disease_information"]
    metadata = {
        "disease": entry["disease"],
        "symptoms": entry["symptoms"],
        "skin_site": entry["skin_site"]
    }
    documents.append(Document(
        page_content=disease_info,
        metadata=metadata
    ))

In [19]:
len(documents) # 질병수

35

### 3. "질병 정보"만 쪼개기

In [20]:
# 3. RecursiveCharacterTextSplitter로 스플리팅
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(documents)


In [21]:
len(split_docs)

175

In [23]:
print(f"총 {len(split_docs)} 개의 청크")
for i, chunk in enumerate(split_docs[:3]):
    print(f"\n청크 {i+1}")
    print(f"질병명: {chunk.metadata['disease']}")
    print(f"증상: {', '.join(chunk.metadata['symptoms'])}")
    print(f"피부 부위: {', '.join(chunk.metadata['skin_site'])}")
    print(f"내용 (앞부분[:300]): {chunk.page_content[:300]} ...")

총 175 개의 청크

청크 1
질병명: Athlete's foot
증상: itchiness, peeling skin, red rash, cracked skin, bleeding
피부 부위: between toes, feet
내용 (앞부분[:300]): Athlete's foot is a type of fungal infection. If your baby has it, you may notice peeling skin between their toes. In more severe cases, the skin on their feet may crack and bleed. The skin can look red on white skin, but this may be less noticeable on brown or black skin. It can also be itchy for y ...

청크 2
질병명: Baby Acne
증상: red spots, pimples, whiteheads
피부 부위: cheeks, forehead, nose
내용 (앞부분[:300]): If your baby has acne, they may have it at birth, but it usually shows up after a couple of weeks. On white skin it looks like small, red spots (pimples), and whiteheads may also develop, sometimes surrounded by reddish skin. On brown and black skin the spots may be harder to see but the skin surrou ...

청크 3
질병명: Baby Acne
증상: red spots, pimples, whiteheads
피부 부위: cheeks, forehead, nose
내용 (앞부분[:300]): Milia happens when a protein called keratin 

### 4. 쪼갠 문서를 임베딩해서 -> 벡터 데이터베이스에 저장

In [24]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain_openai import OpenAIEmbeddings


# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=3072)

In [27]:
# %pip install langchain-pinecone

In [28]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'all-diseases-information-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)


database = PineconeVectorStore.from_documents(split_docs, embedding, index_name=index_name)

c:\Users\BaekSeungJin\rag-practice\.venv\Lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
